In [14]:
def run_python(code_string):
  """
  Pythonコードを文字列で受け取り、実行。実行結果を返す。
  """
  import io
  import contextlib
  import traceback # Tracebackをインポートしてエラー情報を取得できるようにする

  # コードを実行し、標準出力の出力をキャプチャする
  f = io.StringIO()
  try:
    with contextlib.redirect_stdout(f):
        exec(code_string)
  except Exception as _:
    # エラーが発生した場合、そのエラー情報をキャプチャして返す
    return traceback.format_exc()

  # 実行結果を返す
  return f.getvalue()


with open("plan_prompt.txt", encoding="utf-8") as f:
    PLAN_PROMPT = f.read()

with open("code_generator_prompt.txt", encoding="utf-8") as f:
    CODE_GENERATOR_PROMPT = f.read()

with open("graph_prompt.txt", encoding="utf-8") as f:
    GRAPH_GENERATOR_PROMPT = f.read()

with open("summarizer_prompt.txt", encoding="utf-8") as f:
    SUMMARIZER_PROMPT = f.read()

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["font.family"] = "Meiryo"  # Windows

df = pd.read_json("data/qiita_items_2026-05-03_2026-05-31.json")

df = pd.DataFrame({
    "title": df["title"],
    "likes_count": df["likes_count"],
    "comments_count": df["comments_count"],
    "stocks_count": df["stocks_count"],
    "created_at": df["created_at"],
    "tags": df["tags"].apply(lambda x: [tag["name"] for tag in x]),
    "organization": df["organization_url_name"],
    "user_items_count": df["user"].apply(lambda x: x["items_count"]),
    "user_followers_count": df["user"].apply(lambda x: x["followers_count"]),
    "user_followees_count": df["user"].apply(lambda x: x["followees_count"]),
})

df

,title,likes_count,comments_count,stocks_count,created_at,tags,organization,user_items_count,user_followers_count,user_followees_count
0,音韻検索ベンチマークのLLM性能評価関数の整備,0,0,0,2026-05-18 23:53:06+09:00,"[Python, LLM, 音韻検索]",NaN,261,53,1
1,Google検索から“AIに引用される”時代へ。AIOについて考えてみる,0,0,0,2026-05-18 23:50:57+09:00,"[SEO, 考察, フロントエンドエンジニア, AIO]",NaN,27,11,12
2,RAGを使ってAI有識者を作ったら学習がめちゃくちゃ捗った,0,0,0,2026-05-18 23:49:19+09:00,"[rag, duckdb, AIエージェント, ClaudeCode]",NaN,14,9,13
3,LPIC202試験を学ぶ目的を整理（個人メモ）,0,0,0,2026-05-18 23:47:11+09:00,[LPIC202],NaN,160,16,3
4,Best SaaS Admin Dashboard Templates for Modern...,1,0,0,2026-05-18 23:43:40+09:00,"[template, SaaS, React, DashBoard, Next.js]",NaN,24,0,2
...,...,...,...,...,...,...,...,...,...,...
10195,統計学の活用：ロジスティック関数 パラメータとグラフの形状の関係を掴む,0,0,0,2026-05-19 00:26:27+09:00,"[データ分析, 統計学, GeoGebra, データサイエンス, ロジスティック関数]",NaN,281,3,3
10196,Rustのパッケージを配る場所、crates.ioとは何か,1,0,1,2026-05-19 00:25:32+09:00,"[Rust, crates.io]",NaN,39,37,0
10197,技術も経済も分からないバイブコーダーが、トランプ疑惑からAIによる「OSの空洞化」までを浅は...,0,0,1,2026-05-19 00:13:45+09:00,"[Apple, ポエム, AI, OS, LLM]",NaN,17,0,1
10198,非整数階微積分のラプラス変換,0,0,0,2026-05-19 00:08:34+09:00,"[数学, 微分, 積分, 解析学, ラプラス変換]",NaN,85,32,7


In [20]:
import datetime
from pathlib import Path
from typing import TypedDict

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
import json

with open("env.json", encoding="utf-8") as f:
    env = json.load(f)

llm = ChatOpenAI(
    model="gemma-4-12B-it",
    base_url=env["base_url"],
    api_key="dummy"
)

# State定義
class State(TypedDict):
    question: str
    plan: str
    approved: bool
    code: str
    graph_code: str
    result: str
    answer: str


def code_generator(state: State):
    prompt = CODE_GENERATOR_PROMPT.format(plan=state["plan"])

    response = llm.invoke(
        prompt,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    return {
        "code": response.content
    }

def planner(state: State):
    prompt = PLAN_PROMPT.format(question=state["question"])

    response = llm.invoke(prompt)

    return {
        "plan": response.content
    }


def human_approval(state: State):
    # ユーザー確認
    print("分析計画:")
    print(state["plan"])

    approved = input("この計画で実行しますか？ (y/n): ")

    return {
        "approved": approved.lower() == "y"
    }


def executor(state: State):
    result = run_python(
        state["code"]
    )

    return {
        "result": result
    }


def check_error(state: State):
    result = state["result"]

    # tracebackが返ってきた場合
    if "Traceback" in result:
        return "error"

    return "success"

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")


def save_analysis(state: State):
    output_dir = Path("analysis_logs")
    output_dir.mkdir(exist_ok=True)

    filepath = output_dir / f"{timestamp}.md"

    content = (
        "# 分析計画\n\n"
        f"{state['plan']}\n\n"
        "# Pythonコード\n\n"
        "```python\n"
        f"{state['code']}\n"
        "```\n\n"
        "# グラフ作成コード\n\n"
        "```python\n"
        f"{state['graph_code']}\n"
        "```\n"
    )

    filepath.write_text(content, encoding="utf-8")

    print(f"分析内容を保存しました: {filepath}")


filename = Path("analysis_logs") / f"{timestamp}.png"

GRAPH_SAVE_CODE = f"""
plt.savefig(r"{filename}", dpi=200)
plt.close()
"""


def generate_graph(state):
    """
    グラフ生成ノード

    入力:
        state["plan"] : 分析計画
        state["code"] : 分析コード

    出力:
        state["graph_code"] : 生成したグラフコード
    """

    prompt = GRAPH_GENERATOR_PROMPT.format(
        plan=state["plan"],
        code=state["code"]
    )

    response = llm.invoke(
        prompt,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    graph_code = response.content.strip()

    # 分析コード + グラフコード + 保存処理
    full_code = (
        state["code"]
        + "\n\n"
        + graph_code
        + "\n\n"
        + GRAPH_SAVE_CODE
    )

    result = run_python(full_code)

    if "Traceback" in result:
        print(f"グラフ生成に失敗しました。")
        print(result)

    return {
        "graph_code": graph_code
    }


def summarizer(state: State):
    save_analysis(state)
    
    # 集計結果を表示
    print()
    print("# 集計結果")
    print(state["result"])

    prompt = SUMMARIZER_PROMPT.format(question=state["question"], result=state["result"])

    response = llm.invoke(
        prompt,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    return {
        "answer": response.content
    }


graph = StateGraph(State)


graph.add_node(
    "planner",
    planner
)

graph.add_node(
    "human_approval",
    human_approval
)

graph.add_node(
    "code_generator",
    code_generator
)

graph.add_node(
    "executor",
    executor
)

graph.add_node(
    "generate_graph",
    generate_graph
)

graph.add_node(
    "summarizer",
    summarizer
)


graph.set_entry_point(
    "planner"
)


graph.add_edge(
    "planner",
    "human_approval"
)


def approval_router(state: State):
    if state["approved"]:
        return "code_generator"
    return END


graph.add_conditional_edges(
    "human_approval",
    approval_router,
    {
        "code_generator": "code_generator",
        END: END
    }
)

graph.add_edge(
    "code_generator",
    "executor"
)

# TODO:まだエラーを引き継ぐようにしていない。だから、無限に同じミスをし続ける可能性がある。
graph.add_conditional_edges(
    "executor",
    check_error,
    {
        "error": "code_generator",
        "success": "generate_graph"
    }
)

graph.add_edge(
    "generate_graph",
    "summarizer"
)


graph.add_edge(
    "summarizer",
    END
)


app = graph.compile()

In [21]:
result = app.invoke({
    "question": "記事を投稿するのに一番適した時間帯。つまり、一番いいねが付く時間帯。"
})

print(result["answer"])

分析計画:
目的:
記事を投稿した際に、最も多くの「いいね」を獲得できる最適な時間帯（時）および曜日を特定する。

分析方針:
1. 投稿日時から「時間（0〜23時）」と「曜日」を抽出する。
2. 各時間帯および各曜日ごとの「いいね数」の平均値を算出する。
3. 平均いいね数が高い順に時間帯と曜日をランキング化し、最も反応が良い時間帯を特定する。
4. （補足として）投稿数が多い時間帯に偏りがないかを確認するため、投稿数といいね数の相関も考慮する。

具体的な手順:
1. 投稿日時データから、時（Hour）と曜日（Day of Week）の情報を抽出する。
2. 各「時」ごとにグループ化し、その時間帯における「いいね数」の平均値を計算する。
3. 各「曜日」ごとにグループ化し、その曜日における「いいね数」の平均値を計算する。
4. 算出された「時ごとの平均いいね数」と「曜日ごとの平均いいね数」を、値の大きい順に並び替える。
5. 平均いいね数が最大となる「時」と「曜日」を特定する。

出力:
- 平均いいね数が最も高い時間帯（時）とその平均値
- 平均いいね数が最も高い曜日とその平均値
- 時間帯別および曜日別の平均いいね数ランキング（上位5つ）
分析内容を保存しました: analysis_logs\20260722_164739.md

# 集計結果
時間帯別平均いいね数ランキング（上位5つ）:
hour
9     4.632990
4     3.865546
12    3.382576
13    3.189899
11    3.081633
Name: likes_count, dtype: float64

曜日別平均いいね数ランキング（上位5つ）:
day_of_week
Tuesday      2.695592
Monday       2.682781
Thursday     2.449349
Saturday     2.433657
Wednesday    2.125348
Name: likes_count, dtype: float64

分析結果:
平均いいね数が最も高い時間帯: 9時 (平均値: 4.63)
平均いいね数が最も高い曜日: Tuesday (平均値: 2.70)

集計結果に基づくと、最もいいねが付くのは火曜日の

In [22]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	human_approval(human_approval)
	code_generator(code_generator)
	executor(executor)
	generate_graph(generate_graph)
	summarizer(summarizer)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	code_generator --> executor;
	executor -. &nbsp;error&nbsp; .-> code_generator;
	executor -. &nbsp;success&nbsp; .-> generate_graph;
	generate_graph --> summarizer;
	human_approval -.-> __end__;
	human_approval -.-> code_generator;
	planner --> human_approval;
	summarizer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [15]:
plan = """
分析計画:
目的:
記事のタイトル文字数と「いいね数」の相関関係を明らかにし、より多くの「いいね」を得るために適したタイトルの長さ（長文か短文か）を特定する。

分析方針:
- 各記事のタイトル文字数をカウントし、数値データとして抽出する。
- タイトル文字数をいくつかのカテゴリー（例：短文、中編、長文）に分類し、グループごとの「いいね数」の平均値を比較する。
- タイトル文字数と「いいね数」の相関関係を算出して、文字数の増加が「いいね数」に与える影響を定量的に把握する。
- 投稿者のフォロワー数などの影響（バイアス）を考慮に入れるため、グループごとの平均フォロワー数もあわせて確認する。

具体的な手順:
1. 各記事の「記事タイトル」から文字数をカウントし、新しい項目として抽出する。
2. 抽出した文字数に基づき、タイトルを以下の3つのグループに分類する。
   - 短文（例：30文字以下）
   - 中編（例：31文字〜60文字）
   - 長文（例：61文字以上）
3. 各グループごとに、「いいね数」の平均値、中央値、および最大値を算出する。
4. 「タイトル文字数」と「いいね数」の間の相関関係を算出する。
5. 各グループにおける「投稿者のフォロワー数」の平均値を算出し、フォロワー数の差が「いいね数」に与えている影響を把握する。
6. グループごとの「いいね数」の比較結果と相関関係の結果を統合し、結論を導き出す。

出力:
- タイトル文字数グループ別の「いいね数」の平均値および中央値の比較表
- タイトル文字数と「いいね数」の相関係数
- 最も多くの「いいね数」を獲得しているタイトル文字数の傾向（どのグループが最も効果的か）
"""

prompt = CODE_GENERATOR_PROMPT.format(plan=plan)

response = llm.invoke(
    prompt,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
)

print(response.content)

import pandas as pd
import numpy as np

# 1. 各記事のタイトル文字数をカウント
df_temp = df.assign(title_length=df['title'].str.len())

# 2. タイトル文字数に基づき3つのグループに分類
def classify_length(length):
    if length <= 30:
        return '短文'
    elif 31 <= length <= 60:
        return '中編'
    else:
        return '長文'

df_temp = df_temp.assign(length_group=df_temp['title_length'].apply(classify_length))

# 3. 各グループごとに「いいね数」の平均値、中央値、最大値を算出
group_stats = df_temp.groupby('length_group').agg(
    avg_likes=('likes_count', 'mean'),
    median_likes=('likes_count', 'median'),
    max_likes=('likes_count', 'max'),
    avg_followers=('user_followers_count', 'mean')
).reindex(['短文', '中編', '長文'])

print("タイトル文字数グループ別の「いいね数」および「平均フォロワー数」の比較表:")
print(group_stats)
print()

# 4. 「タイトル文字数」と「いいね数」の間の相関関係を算出
correlation = df_temp['title_length'].corr(df_temp['likes_count'])
print(f"タイトル文字数と「いいね数」の相関係数: {correlation:.4f}")
print()

# 6. 結論の導出
best_group = group_stats['avg_likes'].idxmax()
print(f"最も多くの「いいね数」を獲得しているタイトル文字数の傾向:"

In [16]:
print(run_python(response.content))

タイトル文字数グループ別の「いいね数」および「平均フォロワー数」の比較表:
              avg_likes  median_likes  max_likes  avg_followers
length_group                                                   
短文             2.149456           0.0        425      35.090399
中編             2.565180           0.0        576      56.921237
長文             1.606383           0.0        152      65.719605

タイトル文字数と「いいね数」の相関係数: -0.0046

最も多くの「いいね数」を獲得しているタイトル文字数の傾向:
最も効果的なグループ: 中編
そのグループの平均いいね数: 2.57

